In [ ]:
pip install pandas

In [ ]:
import requests
import pandas as pd
from datetime import datetime

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

start_year = datetime.now().year - 5
end_year = datetime.now().year

all_records = []

for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        start_date = f"{year}-{month:02d}-01"
        if month == 12:
            end_date = f"{year+1}-01-01"
        else:
            end_date = f"{year}-{month+1:02d}-01"

        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date,
            "minmagnitude": 3
        }
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f" Failed for {start_date}: {response.text[:200]}")
            continue

        try:
            data = response.json()
        except Exception as e:
            print(f" JSON error for {start_date}: {e}")
            continue

        for f in data["features"]:
            p=f["properties"]
            g=f["geometry"]["coordinates"]
            all_records.append({
                "id": f.get("id"),
                "time": pd.to_datetime(p.get("time"), unit="ms"),
                "updated": pd.to_datetime(p.get("updated"), unit="ms"),
                "latitude": g[1] if g else None,
                "longitude": g[0] if g else None,
                "depth_km": g[2] if g else None,
                "mag": p.get("mag"),
                "magType": p.get("magType"),
                "place": p.get("place"),
                "status": p.get("status"),
                "tsunami": p.get("tsunami"),
                "alert": p.get("alert"),
                "felt": p.get("felt"),
                "cdi": p.get("cdi"),
                "mmi": p.get("mmi"),
                "sig": p.get("sig"),
                "net": p.get("net"),
                "code": p.get("code"),
                "ids": p.get("ids"),
                "sources": p.get("sources"),
                "types": p.get("types"),
                "nst": p.get("nst"),
                "dmin": p.get("dmin"),
                "rms": p.get("rms"),
                "gap": p.get("gap"),
                "type": p.get("type")
            })
df=pd.DataFrame(all_records)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df.head())


Rows: 121771
Columns: 26
           id                    time                 updated  latitude  \
0  us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   28.1524   
4  us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag magType  \
0   -68.9337     17.27  4.7     mwr   
1  -177.2052    426.71  4.1      mb   
2   121.3159     46.73  4.7      mb   
3    57.2570     10.00  4.9      mb   
4    -3.7578     10.00  4.0      mb   

                                               place    status  ...  net  \
0        29 km SW of Villa Basilio Nievas, Argentina  reviewed  ...   us   
1                                        Fiji region  reviewed  ...   us   
2                    103 km SW of Basco, Philippines  reviewe

In [ ]:
df.columns

Index(['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag',
       'magType', 'place', 'status', 'tsunami', 'alert', 'felt', 'cdi', 'mmi',
       'sig', 'net', 'code', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms',
       'gap', 'type'],
      dtype='object')

In [ ]:
print(df)

                id                    time                 updated   latitude  \
0       us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040 -31.749300   
1       us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040 -15.490200   
2       us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040  19.752900   
3       us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040  28.152400   
4       us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040  71.321200   
...            ...                     ...                     ...        ...   
121766  us6000typ0 2026-10-01 01:31:19.148 2026-10-22 01:05:14.040  -0.826300   
121767  us6000tyny 2026-10-01 01:29:20.875 2026-10-22 00:39:24.864  36.107800   
121768  us6000tynt 2026-10-01 00:41:02.788 2026-10-21 01:27:03.040 -36.254300   
121769  us6000tynp 2026-10-01 00:28:22.215 2026-10-21 01:08:25.040  19.230700   
121770  ci41341279 2026-10-01 00:01:59.630 2026-10-21 00:56:54.040  33.502333   

         longitude  depth_k

In [ ]:
import io, time

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
extra_frames = []

for year in range(datetime.now().year - 5, datetime.now().year + 1):
    for month in range(1, 13):
        start_date = f"{year}-{month:02d}-01"
        end_date = f"{year+1}-01-01" if month == 12 else f"{year}-{month+1:02d}-01"
        if pd.Timestamp(start_date) > pd.Timestamp.today():
            continue
        params = {"format": "csv", "starttime": start_date,
                  "endtime": end_date, "minmagnitude": 3}
        for attempt in range(3):
            r = requests.get(url, params=params, timeout=90)
            if r.status_code in (200, 204):
                break
            time.sleep(3)
        else:
            print("Failed:", start_date, r.status_code)
            continue
        if r.status_code == 204 or not r.text.strip():
            continue
        extra_frames.append(pd.read_csv(io.StringIO(r.text),
            usecols=["id","magError","depthError","magNst","locationSource","magSource"]))
        print(start_date, len(extra_frames[-1]))

extra = pd.concat(extra_frames).drop_duplicates(subset="id")
df = df.merge(extra, on="id", how="left")
print(df.shape)     # rows must stay 121710

2021-01-01 1714
2021-02-01 1815
2021-03-01 2437
2021-04-01 1577
2021-05-01 1604
2021-06-01 1613
2021-07-01 1801
2021-08-01 2683
2021-09-01 1688
2021-10-01 1536
2021-11-01 1543
2021-12-01 1915
2022-01-01 1910
2022-02-01 1615
2022-03-01 1917
2022-04-01 1657
2022-05-01 1507
2022-06-01 1530
2022-07-01 1693
2022-08-01 1681
2022-09-01 1903
2022-10-01 1618
2022-11-01 1644
2022-12-01 1533
2023-01-01 1787
2023-02-01 1852
2023-03-01 1634
2023-04-01 1548
2023-05-01 1851
2023-06-01 1644
2023-07-01 1559
2023-08-01 1498
2023-09-01 1453
2023-10-01 1738
2023-11-01 1752
2023-12-01 2514
2024-01-01 1734
2024-02-01 1529
2024-03-01 1446
2024-04-01 1841
2024-05-01 1608
2024-06-01 1435
2024-07-01 1444
2024-08-01 1426
2024-09-01 1386
2024-10-01 1525
2024-11-01 1437
2024-12-01 1848
2025-01-01 1745
2025-02-01 1712
2025-03-01 1634
2025-04-01 1688
2025-05-01 1586
2025-06-01 1499
2025-07-01 3328
2025-08-01 2569
2025-09-01 1901
2025-10-01 1728
2025-11-01 1578
2025-12-01 1865
2026-01-01 1688
2026-02-01 1542
2026-03-

In [ ]:
# Merge 'extra' columns back into df if they are missing
missing_cols = [c for c in ["magError","depthError","magNst","locationSource","magSource"] if c not in df.columns]
if missing_cols and 'extra' in globals():
    df = df.merge(extra[["id"] + missing_cols].drop_duplicates(subset="id"), on="id", how="left")

features = ["id","time","updated","latitude","longitude","depth_km","mag","magType","place",
            "status","tsunami","sig","net","nst","dmin","rms","gap","magError","depthError",
            "magNst","locationSource","magSource","types","ids","sources","type"]

df = df[features]
print(df.shape)     # expect (121710, 26)
df.to_csv("raw_earthquakes.csv", index=False)

(121771, 26)


## Data Cleaning

In [ ]:
import pandas as pd
df = pd.read_csv("raw_earthquakes.csv", parse_dates=["time", "updated"])
df = df.drop_duplicates(subset="id").reset_index(drop=True)
print("Shape:", df.shape)
print(df.isna().sum())          # null counts BEFORE cleaning

Shape: (121771, 26)
id                    0
time                  0
updated               0
latitude              0
longitude             0
depth_km              0
mag                   0
magType               0
place                 0
status                0
tsunami               0
sig                   0
net                   0
nst               27762
dmin               3980
rms                  20
gap                3025
magError           4123
depthError            4
magNst             3489
locationSource        0
magSource             0
types                 0
ids                   0
sources               0
type                  0
dtype: int64


In [ ]:
numeric_cols = ["latitude","longitude","depth_km","mag","tsunami","sig","nst","dmin",
                "rms","gap","magError","depthError","magNst"]
categorical_cols = ["id","magType","place","status","net","locationSource","magSource",
                    "types","ids","sources","type"]

for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(df[c].mean())

for c in ["tsunami","sig","nst","magNst"]:      # count columns: keep whole numbers
    df[c] = df[c].round().astype(int)

for c in categorical_cols:
    df[c] = df[c].astype(str).str.strip().replace("nan", pd.NA)
    df[c] = df[c].fillna(df[c].mode()[0])

In [ ]:
print(df.isna().sum())          # every column must be 0
print(df.shape)                 # (about 121,710, 26)
df.to_csv("cleaned_earthquakes.csv", index=False)

from google.colab import files
files.download("cleaned_earthquakes.csv")

id                0
time              0
updated           0
latitude          0
longitude         0
depth_km          0
mag               0
magType           0
place             0
status            0
tsunami           0
sig               0
net               0
nst               0
dmin              0
rms               0
gap               0
magError          0
depthError        0
magNst            0
locationSource    0
magSource         0
types             0
ids               0
sources           0
type              0
dtype: int64
(121771, 26)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import re

US = {"AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC",
      "Alaska","Hawaii","California","Nevada","Oregon","Washington","Oklahoma","Texas","Utah","Idaho","Montana","Wyoming","Colorado","New Mexico","Arizona","Kansas","Tennessee","Missouri"}

ALIAS = {"Burma (Myanmar)":"Myanmar", "Türkiye":"Turkey", "Macedonia":"North Macedonia",
         "Fiji Islands":"Fiji", "Fiji region":"Fiji",
         "Japan region":"Japan", "Tonga region":"Tonga"}

def get_country(place):
    m = re.search(r",\s*([^,]+)$", place)               # text after the last comma
    if m:
        name = m.group(1).strip()
    else:                                               # e.g. "south of the Fiji Islands"
        name = re.sub(r"^(?:north|south|east|west|northeast|northwest|southeast|southwest)\s+of\s+(?:the\s+)?", "", place, flags=re.I)
    name = re.sub(r"\s+region$", "", name, flags=re.I).strip()   # now applies to both cases
    if name in US:
        return "United States"
    return ALIAS.get(name, name)

df["country"] = df["place"].apply(get_country)
print(df["country"].nunique())
print(df["country"].value_counts().head(25))

448
country
United States               21348
Indonesia                    9707
Japan                        7364
Russia                       6559
Philippines                  5412
Fiji                         4844
Papua New Guinea             3905
Tonga                        3533
Puerto Rico                  3521
Chile                        3303
South Sandwich Islands       3290
Kermadec Islands             2812
Vanuatu                      2400
New Zealand                  1803
Mexico                       1690
U.S. Virgin Islands          1664
Argentina                    1588
China                        1560
Greece                       1523
Northern Mariana Islands     1430
Peru                         1410
Dominican Republic           1313
Solomon Islands              1209
Taiwan                       1115
Loyalty Islands              1106
Name: count, dtype: int64


In [ ]:
PATCH = {"Timor Leste": "Timor-Leste",
         "Kuril Islands": "Russia",
         "Mariana Islands": "Northern Mariana Islands",
         "off the coast of Oregon": "United States",
         "MX": "Mexico",
         "western Xizang": "China",
         "Macquarie Island": "Australia",
         "Easter Island": "Chile"}
df["country"] = df["country"].replace(PATCH)

# check for any other leftover state/country codes (2 letters)
print(df[df["country"].str.len() <= 2]["country"].value_counts())

Series([], Name: count, dtype: int64)


In [ ]:
!pip -q install pycountry_convert
import pycountry_convert as pc

MANUAL = {"Kosovo":"Europe", "Taiwan":"Asia", "United States":"North America", "Russia":"Europe",
          "Turkey":"Asia", "Myanmar":"Asia", "Democratic Republic of the Congo":"Africa",
          "Puerto Rico":"North America", "U.S. Virgin Islands":"North America",
          "Northern Mariana Islands":"Oceania", "Kermadec Islands":"Oceania",
          "Loyalty Islands":"Oceania", "New Caledonia":"Oceania",
          "Timor-Leste":"Asia", "Africa":"Africa", "Ascension Island":"Africa",
          "off the coast of Central America":"North America",
          "South Shetland Islands":"Antarctica",
          "South Sandwich Islands":"Ocean/Other"}

def get_continent(c):
    if c in MANUAL: return MANUAL[c]
    try:
        code = pc.country_name_to_country_alpha2(c)
        return pc.convert_continent_code_to_continent_name(pc.country_alpha2_to_continent_code(code))
    except Exception:
        return "Ocean/Other"

lookup = {c: get_continent(c) for c in df["country"].unique()}
df["continent"] = df["country"].map(lookup)
print(df["continent"].value_counts())
print(df[df.continent == "Ocean/Other"]["country"].value_counts().head(30))

continent
North America    32999
Asia             32535
Oceania          25791
Ocean/Other      11068
Europe            9706
South America     7985
Africa            1556
Antarctica         131
Name: count, dtype: int64
country
South Sandwich Islands            3290
Reykjanes Ridge                    649
Mid-Indian Ridge                   590
Banda Sea                          559
northern Mid-Atlantic Ridge        486
southern Mid-Atlantic Ridge        468
central Mid-Atlantic Ridge         371
Pacific-Antarctic Ridge            276
western Indian-Antarctic Ridge     262
West Chile Rise                    241
Carlsberg Ridge                    236
southern East Pacific Rise         232
southeast Indian Ridge             204
Southwest Indian Ridge             197
central East Pacific Rise          169
Balleny Islands                    144
Chagos Archipelago                  96
South Indian Ocean                  90
Owen Fracture Zone                  85
Saint Helena                   

In [ ]:
# 1. country fixes (also catches other variants such as "Northern Turkey")
df["country"] = df["country"].replace({"Severnaya Zemlya": "Russia"})
df.loc[df["country"].str.contains("Turkey", case=False), "country"] = "Turkey"
df.loc[df["country"].str.contains("Tibet|Xizang", case=False), "country"] = "China"

# 2. continent fixes
MANUAL.update({"Saint Helena": "Africa", "Prince Edward Islands": "Africa",
               "Virgin Islands": "North America", "Turkey": "Asia", "China": "Asia"})
lookup = {c: get_continent(c) for c in df["country"].unique()}
df["continent"] = df["country"].map(lookup)

print(df["continent"].value_counts())
print(df[df.continent == "Ocean/Other"]["country"].value_counts().iloc[20:60])

continent
North America    33063
Asia             32706
Oceania          25791
Ocean/Other      10618
Europe            9778
South America     7985
Africa            1699
Antarctica         131
Name: count, dtype: int64
country
Drake Passage                                            76
Mona Passage                                             73
Molucca Sea                                              72
Greenland Sea                                            69
North Island of New Zealand                              51
Revilla Gigedo Islands                                   51
Indian Ocean Triple Junction                             49
northern East Pacific Rise                               48
Alaska Peninsula                                         42
Minnesota                                                41
Galapagos Triple Junction                                39
Mauritius - Reunion                                      38
Norwegian Sea                                       

In [ ]:
PATCH2 = {
    # United States
    "Alaska Peninsula": "United States", "Gulf of Alaska": "United States",
    "Southern Alaska": "United States", "Central Alaska": "United States",
    "Minnesota": "United States", "western Texas": "United States",
    "Louisiana": "United States", "off the coast of Washington": "United States",
    # New Zealand
    "North Island of New Zealand": "New Zealand",
    "off the east coast of the North Island of New Zealand": "New Zealand",
    # Other countries
    "off the coast of Ecuador": "Ecuador", "Galapagos Islands": "Ecuador",
    "off the west coast of northern Sumatra": "Indonesia",
    "near the coast of Nicaragua": "Nicaragua",
    "southern Iran": "Iran",
    "Chile-Argentina border": "Chile", "Chile-Bolivia border": "Chile",
    "Philippine Islands": "Philippines",
    "Samoa Islands": "Samoa",
    "Santa Cruz Islands": "Solomon Islands",
    "D'Entrecasteaux Islands": "Papua New Guinea",
    "Revilla Gigedo Islands": "Mexico",
    "Azores Islands": "Portugal",
    "Mauritius - Reunion": "Mauritius",
}
df["country"] = df["country"].replace(PATCH2)

MANUAL.update({"Leeward Islands": "North America",
               "Saint Eustatius and Saba": "North America",
               "Tristan da Cunha": "Africa"})
lookup = {c: get_continent(c) for c in df["country"].unique()}
df["continent"] = df["country"].map(lookup)

print(df["continent"].value_counts())
print(df[df.continent == "Ocean/Other"]["country"].value_counts().head(40))
print(df["country"].value_counts().head(5))     # United States should now be larger

continent
North America    33391
Asia             32772
Oceania          25939
Ocean/Other       9889
Europe            9804
South America     8083
Africa            1762
Antarctica         131
Name: count, dtype: int64
country
South Sandwich Islands               3290
Reykjanes Ridge                       649
Mid-Indian Ridge                      590
Banda Sea                             559
northern Mid-Atlantic Ridge           486
southern Mid-Atlantic Ridge           468
central Mid-Atlantic Ridge            371
Pacific-Antarctic Ridge               276
western Indian-Antarctic Ridge        262
West Chile Rise                       241
Carlsberg Ridge                       236
southern East Pacific Rise            232
southeast Indian Ridge                204
Southwest Indian Ridge                197
central East Pacific Rise             169
Balleny Islands                       144
Chagos Archipelago                     96
South Indian Ocean                     90
Owen Fracture Zo

In [ ]:
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["day_of_week"] = df["time"].dt.day_name()
df["hour"] = df["time"].dt.hour

df["depth_category"] = pd.cut(df["depth_km"], [-1000, 70, 300, 1000],
        labels=["Shallow","Intermediate","Deep"], right=False).astype(str)
df["mag_category"] = pd.cut(df["mag"], [0,4,5,6,7,8,11],
        labels=["Minor","Light","Moderate","Strong","Major","Great"], right=False).astype(str)
df["is_strong"] = (df["mag"] >= 6).astype(int)

print(df.shape)                      # expect (121771, 36)
print(df.isna().sum().sum())         # expect 0
print(df["depth_category"].value_counts())
print(df["mag_category"].value_counts())

(121771, 36)
0
depth_category
Shallow         91167
Intermediate    23543
Deep             7061
Name: count, dtype: int64
mag_category
Light       83829
Minor       26925
Moderate    10225
Strong        704
Major          84
Great           4
Name: count, dtype: int64


In [ ]:
df.to_csv("cleaned_earthquakes.csv", index=False)
from google.colab import files
files.download("cleaned_earthquakes.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>